# Vulnerability Report Enhancement — tianocore/edk2

A **triager-assist** tool: fetches published **repository security advisories** from the edk2 [Security tab](https://github.com/tianocore/edk2/security), sends each to **claude-sonnet-4-6** to confirm or correct the metadata GitHub already has (severity, affected package) and to extract what it doesn't structure separately (component, build target, architecture, impact, patch complexity, criticality) — then separately fetches **ground truth from GitHub** (repo structure, linked fix commits/PRs) to check how often the result lines up with something real, so a triager knows how much to trust it.

**Pipeline:** Fetch advisories → LLM review (confirm/correct + extract) → GitHub ground truth → Compare → Evaluate → Export

The CSV/JSON export keeps three things distinct for every field that can be checked:
- **`llm_*`** — exactly what the model returned, unmodified (its confirmation/correction of what GitHub reported, or its extraction from context where nothing was reported)
- **`ground_truth_*`** / **`fix_*`** — what's actually true, fetched from GitHub (the advisory's own reported package, repo tree structure, `edk2-platforms`, or the real fix commit/PR diff when the advisory links one)
- **`*_match`** — whether the two agree, wherever a ground truth exists to compare against


In [13]:
# Cell 1: Install dependencies
!pip install anthropic requests --quiet
print("Dependencies installed")

Dependencies installed


## Step 1 — API Keys

Keys are loaded from **Colab Secrets** (the key icon in the left sidebar).

| Secret name | What it is |
|---|---|
| `ANTHROPIC_API_KEY` | Your Anthropic key — https://console.anthropic.com |
| `GITHUB_TOKEN` | GitHub personal access token (**strongly recommended** — this notebook makes many GitHub API calls: advisories, repo trees, and per-advisory commit/PR diffs) — https://github.com/settings/tokens |

> **How to add a secret:** Click the key icon in the left sidebar → *Add new secret* → paste the value → toggle *Notebook access* ON.


In [14]:
# Cell 2: Load API keys from Colab Secrets
import os
from google.colab import userdata

def get_secret(name):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(name, "")

ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY")
GITHUB_TOKEN  = get_secret("GITHUB_TOKEN")

print("ANTHROPIC_API_KEY:", "OK (set)" if ANTHROPIC_KEY else "MISSING — add it in Colab Secrets (key icon in sidebar)")
print("GITHUB_TOKEN     :", "OK (set)" if GITHUB_TOKEN  else "not set — unauthenticated (60 req/hr limit)")

if not ANTHROPIC_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY not found.\n"
        "Go to the key icon in the left sidebar → Add new secret:\n"
        "  Name: ANTHROPIC_API_KEY   Value: sk-ant-...\n"
        "Then toggle 'Notebook access' ON and re-run this cell."
    )

ANTHROPIC_API_KEY: OK (set)
GITHUB_TOKEN     : OK (set)


## Step 2 — Configuration

In [15]:
# Cell 3: Configuration
# @title Configuration { run: "auto" }
GITHUB_REPO      = 'tianocore/edk2'
PLATFORMS_REPO   = 'tianocore/edk2-platforms'   # separate repo — where real hardware/vendor/SoC dirs actually live
ADVISORY_STATE   = 'published'
GITHUB_API       = 'https://api.github.com'
MODEL_NAME       = 'claude-sonnet-4-6'
GROUND_AGAINST_REPO = True   # @param {type:"boolean"}  fetch real repo structure + fix diffs as ground truth

# Advisories to skip entirely — never enhanced, never fetched for ground truth, never counted in evaluation.
# GHSA-5xcf-j538-p769: placeholder/test-like entry (CVE-2024-00000 is not a real CVE ID; sparse structured data).
EXCLUDED_GHSA_IDS = {'GHSA-5xcf-j538-p769'}

MAX_ADVISORIES = 20    # @param {type:"slider", min:5, max:100, step:5}
AI_DELAY_SEC   = 0.5   # @param {type:"number"}
RUN_AI         = True  # @param {type:"boolean"}

# Valid options for structured / scored fields (VALID_PACKAGES gets replaced with the real repo listing in Step 6
# if GROUND_AGAINST_REPO is on — this is the fallback if that fetch fails or is disabled)
VALID_PACKAGES = [
    'ArmPkg', 'ArmPlatformPkg', 'ArmVirtPkg', 'BaseTools', 'Build or CI Code',
    'CryptoPkg', 'DynamicTablesPkg', 'EmbeddedPkg', 'EmulatorPkg', 'FatPkg',
    'FmpDevicePkg', 'IntelFsp2Pkg', 'IntelFsp2WrapperPkg', 'ManageabilityPkg',
    'MdeModulePkg', 'MdePkg', 'NetworkPkg', 'OvmfPkg', 'PcAtChipsetPkg',
    'PrmPkg', 'RedfishPkg', 'SecurityPkg', 'ShellPkg', 'SourceLevelDebugPkg',
    'StandaloneMmPkg', 'UefiCpuPkg', 'UefiPayloadPkg', 'UnitTestFrameworkPkg', 'Other'
]
VALID_BUILD_TARGET  = ['DEBUG', 'RELEASE', 'NOOPT', 'NO-TARGET', 'None']
VALID_ARCHITECTURES = ['IA32', 'X64', 'AARCH64', 'ARM', 'RISCV64', 'LOONGARCH64']
# Matches GitHub's own Severity dropdown exactly. 'Unknown/Pending selection' isn't itself a dropdown value,
# so it's represented as an empty severity_score rather than a VALID_SEVERITY entry.
VALID_SEVERITY = ['Low', 'Moderate', 'High', 'Critical', 'Assess severity using CVSS v3', 'Assess severity using CVSS v4']
VALID_PATCH_COMPLEXITY    = ['simple', 'moderate', 'complex']
VALID_CRITICALITY = ['low', 'medium', 'high']

# Thresholds for classifying real fix-diff size into a patch_complexity — tune if 'moderate' feels miscalibrated
PATCH_COMPLEXITY_THRESHOLDS = {'simple_max_files': 1, 'simple_max_lines': 30, 'moderate_max_files': 5, 'moderate_max_lines': 150}

print(f'Config: model={MODEL_NAME}, {MAX_ADVISORIES} advisories, state={ADVISORY_STATE}, AI={"on" if RUN_AI else "off"}')

Config: model=claude-sonnet-4-6, 20 advisories, state=published, AI=on


## Step 3 — Data model

In [16]:
# Cell 4: Data model
import re, json, time
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class AdvisoryEnhancement:
    # ---- Raw GHSA fields (from GitHub's repository security-advisories API) ----
    ghsa_id:            str
    cve_id:             Optional[str]
    url:                str
    original_summary:   str
    description:        str
    github_severity:    str            # GitHub's own rating: '' (unrated) | Low | Moderate | High | Critical
    github_severity_source: str        # 'reported' (GH's own flat field) | 'derived-from-cvss' | '' (genuinely unrated)
    cvss_score:         Optional[float]
    cvss_vector:        str
    cvss_version:       str            # 'v3' | 'v4' | 'legacy' (deprecated flat field) | '' (no CVSS at all)
    cwe_ids:            list
    affected_packages:  list
    references:         list           # raw reference URLs — may include the actual fix commit/PR
    published_at:       str

    # ---- LLM prediction (Step 8) — exactly what the model said, never overwritten by ground truth ----
    llm_package:                    list  = field(default_factory=list)
    llm_package_confirmed:          str   = ''   # Yes | No | Not reported — LLM's confirmation of GitHub's reported package
    llm_component:                  str   = ''
    llm_build_target:               list  = field(default_factory=list)
    llm_architecture:               list  = field(default_factory=list)
    llm_affected_devices:           str   = ''
    llm_impact:                     str   = ''
    llm_severity_confirmed:         str   = ''   # Yes | No | Uncertain — LLM's opinion on github_severity
    llm_severity_reason:            str   = ''
    llm_severity_override:          str   = ''   # LLM's suggested rating if it disagreed (or GH was unrated)
    llm_patch_complexity:             str   = ''
    llm_patch_complexity_reason:            str   = ''
    llm_criticality_score:          str   = ''
    llm_criticality_reason:         str   = ''
    llm_reflective_confidence:      int   = 0
    llm_reflective_confidence_reason: str = ''

    # Per-field confidence (0-100) — how well-supported EACH individual LLM-based addition is by the advisory
    # text, independent of the overall reflective_confidence above (which covers the enhancement as a whole).
    llm_package_confidence:          int = 0
    llm_component_confidence:        int = 0
    llm_build_target_confidence:     int = 0
    llm_architecture_confidence:     int = 0
    llm_affected_devices_confidence: int = 0
    llm_impact_confidence:           int = 0
    llm_severity_confidence:         int = 0
    llm_patch_complexity_confidence:         int = 0
    llm_criticality_confidence:      int = 0

    # ---- Ground truth fetched from GitHub (Step 6 = repo/platforms tree, Step 7 = linked fix commit/PR) ----
    fix_ref_url:                str = ''
    fix_files_changed:          Optional[int] = None
    fix_lines_changed:          Optional[int] = None
    fix_file_paths:             list = field(default_factory=list)
    ground_truth_package:      list = field(default_factory=list)   # from fix diff only — independent, not a guess
    ground_truth_package_source: str = ''   # 'ghsa-reported' (advisory's own affected-product field) | 'fix-diff' (more precise, overrides)
    ground_truth_component:    str  = ''                              # from GHSA's reported product (if it named a module) or fix diff
    ground_truth_component_source: str = ''   # 'ghsa-reported' | 'fix-diff' (more precise, overrides)
    ground_truth_patch_complexity: str = ''                             # from fix diff size, threshold-classified

    # ---- Comparison: does the LLM prediction agree with GitHub ground truth? None = no ground truth to check ----
    package_match:    Optional[bool] = None
    component_match:  Optional[bool] = None
    patch_complexity_match:   Optional[bool] = None
    severity_match:   Optional[bool] = None   # llm_severity_confirmed == 'Yes', only when GitHub had a rating to confirm

    # ---- Weaker plausibility checks against real repo listings, used only when there's no independent ground truth ----
    package_valid:  Optional[bool] = None   # every llm_package entry is a real top-level dir
    component_valid: Optional[bool] = None  # llm_component fuzzy-matches a real path in the edk2 tree
    device_valid:   Optional[bool] = None   # llm_affected_devices contains a known edk2-platforms vendor/board token

    # ---- Resolved 'best answer' fields — ground truth if available, else the LLM prediction. For convenience only;
    #      the actual comparison lives in the fields above. ----
    package:          list = field(default_factory=list)
    package_source:   str  = ''   # 'fix-diff' | 'llm'
    component:        str  = ''
    component_source: str  = ''
    build_target:     list = field(default_factory=list)
    architecture:      list = field(default_factory=list)
    affected_devices: str  = ''
    impact:           str  = ''
    severity_score:      str = ''   # from github_severity, or llm_severity_override if it disagreed
    patch_complexity:       str = ''
    patch_complexity_source:      str = ''
    patch_complexity_reason:      str = ''
    criticality_score:    str = ''
    criticality_reason:   str = ''

    validation_warnings: list = field(default_factory=list)
    ai_error: Optional[str] = None

def cvss_score_to_severity(score):
    """Standard CVSS v3.x/v4 qualitative severity scale (the same buckets GitHub's UI uses to render a label)."""
    if score is None:
        return ''
    if score < 4.0:
        return 'Low'
    if score < 7.0:
        return 'Moderate'
    if score < 9.0:
        return 'High'
    return 'Critical'

def prepare_advisory(adv):
    vulns = adv.get('vulnerabilities') or []
    pkgs = [
        {
            'ecosystem':                (v.get('package') or {}).get('ecosystem', ''),
            'name':                     (v.get('package') or {}).get('name', ''),
            'vulnerable_version_range': v.get('vulnerable_version_range', ''),
            'patched_versions':         v.get('patched_versions', ''),
        }
        for v in vulns
    ]
    cwes = [c.get('cwe_id', '') for c in (adv.get('cwes') or [])]
    # GitHub deprecated the flat 'cvss' field in favor of 'cvss_severities' (removed from the REST API entirely
    # on 2025-04-01) — read the new field first, falling back to the old one only for any advisory still carrying
    # legacy data there. Prefer v3 over v4 as the more universally-supported baseline (matches VALID_SEVERITY's
    # own v3-before-v4 default elsewhere).
    # NOTE: cvss_severities always contains cvss_v3/cvss_v4 sub-dicts, even when unassessed (score: null,
    # vector_string: '') — check the actual score value, not dict truthiness, or every advisory looks like v3.
    cvss_severities = adv.get('cvss_severities') or {}
    v3 = cvss_severities.get('cvss_v3') or {}
    v4 = cvss_severities.get('cvss_v4') or {}
    legacy = adv.get('cvss') or {}
    if v3.get('score') is not None:
        cvss, cvss_version = v3, 'v3'
    elif v4.get('score') is not None:
        cvss, cvss_version = v4, 'v4'
    elif legacy.get('score') is not None:
        cvss, cvss_version = legacy, 'legacy'
    else:
        cvss, cvss_version = {}, ''
    refs = [r.get('url', '') for r in (adv.get('references') or []) if isinstance(r, dict) and r.get('url')]
    gh_severity_raw = (adv.get('severity') or '').strip().lower()
    gh_severity = {'low': 'Low', 'moderate': 'Moderate', 'high': 'High', 'critical': 'Critical'}.get(gh_severity_raw, '')
    github_severity_source = 'reported' if gh_severity else ''
    if not gh_severity and cvss.get('score') is not None:
        # GitHub computes and displays a severity label from the CVSS score when the advisory was rated via
        # 'Assess severity using CVSS v3/v4' rather than a manually-picked level — but its API's flat 'severity'
        # field comes back null for those. Derive the same label using the standard CVSS qualitative scale so we
        # don't silently lose the rating for every CVSS-assessed advisory (which is most of them in practice).
        gh_severity = cvss_score_to_severity(cvss['score'])
        github_severity_source = 'derived-from-cvss'
    return AdvisoryEnhancement(
        ghsa_id           = adv.get('ghsa_id', ''),
        cve_id            = adv.get('cve_id'),
        url               = adv.get('html_url', ''),
        original_summary  = (adv.get('summary') or '').strip(),
        description       = (adv.get('description') or '').strip(),
        github_severity   = gh_severity,
        github_severity_source = github_severity_source,
        cvss_score        = cvss.get('score'),
        cvss_version      = cvss_version,
        cvss_vector       = cvss.get('vector_string', '') or '',
        cwe_ids           = cwes,
        affected_packages = pkgs,
        references        = refs,
        published_at      = adv.get('published_at', '') or '',
    )

print('Data model defined')

Data model defined


## Step 4 — Fetch published security advisories from GitHub

In [17]:
# Cell 5: Fetch security advisories
import requests

def validate_github_token(token):
    if not token:
        return False, "no token provided"
    resp = requests.get(f"{GITHUB_API}/user", headers={"Accept": "application/vnd.github+json",
                         "Authorization": f"Bearer {token}"}, timeout=10)
    if resp.status_code == 200:
        return True, resp.json().get("login", "unknown")
    elif resp.status_code == 401:
        return False, "token is invalid or expired (401)"
    return False, f"unexpected status {resp.status_code}"

def fetch_advisories(token, max_advisories):
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    advisories, page = [], 1
    per_page = min(100, max_advisories)
    while len(advisories) < max_advisories:
        url = (f"{GITHUB_API}/repos/{GITHUB_REPO}/security-advisories"
               f"?state={ADVISORY_STATE}&sort=published&direction=desc&per_page={per_page}&page={page}")
        resp = requests.get(url, headers=headers, timeout=30)
        if resp.status_code == 401:
            raise RuntimeError("GitHub 401 Unauthorized — check/refresh GITHUB_TOKEN in Colab Secrets, or clear it to go unauthenticated.")
        if resp.status_code == 403:
            raise RuntimeError("GitHub 403: rate limit hit. Add or refresh your GITHUB_TOKEN.")
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        advisories.extend(batch)
        page += 1
        if len(batch) < per_page:
            break
    return advisories[:max_advisories]

token_valid, token_info = validate_github_token(GITHUB_TOKEN)
if GITHUB_TOKEN:
    if token_valid:
        print(f"GitHub token OK — authenticated as: {token_info}")
    else:
        print(f"WARNING: GitHub token check failed — {token_info}")
        print("Continuing without auth (60 req/hr limit)...")
        GITHUB_TOKEN = ""
else:
    print("No GitHub token — fetching unauthenticated (60 req/hr limit)")

print(f"Fetching up to {MAX_ADVISORIES} {ADVISORY_STATE!r} security advisories from {GITHUB_REPO}...")
raw_advisories = fetch_advisories(GITHUB_TOKEN, MAX_ADVISORIES)
print(f"Fetched {len(raw_advisories)} advisories")

GitHub token OK — authenticated as: nazaninsiavash
Fetching up to 20 'published' security advisories from tianocore/edk2...
Fetched 13 advisories


## Step 5 — Prepare advisories

Anything in `EXCLUDED_GHSA_IDS` (config cell) is dropped here, before it touches the LLM, ground-truth fetches, or the evaluation — not filtered out afterward.


In [18]:
# Cell 6: Prepare advisories
excluded_count = sum(1 for adv in raw_advisories if adv.get('ghsa_id') in EXCLUDED_GHSA_IDS)
results = [prepare_advisory(adv) for adv in raw_advisories if adv.get('ghsa_id') not in EXCLUDED_GHSA_IDS]
if excluded_count:
    print(f'Excluded {excluded_count} advisory(ies) per EXCLUDED_GHSA_IDS: {sorted(EXCLUDED_GHSA_IDS)}')
print(f'Prepared {len(results)} advisories')
print(f'Advisories with a CVE ID   : {sum(1 for r in results if r.cve_id)}')
print(f'Advisories with CVSS score : {sum(1 for r in results if r.cvss_score is not None)}')

Excluded 1 advisory(ies) per EXCLUDED_GHSA_IDS: ['GHSA-5xcf-j538-p769']
Prepared 12 advisories
Advisories with a CVE ID   : 12
Advisories with CVSS score : 12


## Step 6 — Fetch GitHub repo structure (ground truth source #1)

Fetches the real directory listings this notebook checks predictions against later:
- `tianocore/edk2`'s own tree → real top-level `*Pkg` directories (grounds `VALID_PACKAGES`) and real `Package/Module` paths (checks `llm_component` plausibility)
- `tianocore/edk2-platforms`'s tree → real vendor/board directories (checks `llm_affected_devices` plausibility)

This is a **plausibility** check (is this a real thing in the codebase), not independent ground truth for a *specific* advisory — that stronger check comes from the linked fix commit/PR in Step 7, when one exists.


In [19]:
# Cell 6b: Fetch real repo structure
import difflib
from collections import Counter

def resolve_default_branch(owner_repo, headers):
    resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}", headers=headers, timeout=15)
    return resp.json().get('default_branch', 'master') if resp.status_code == 200 else 'master'

def fetch_repo_tree(owner_repo, token):
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    branch = resolve_default_branch(owner_repo, headers)
    ref_resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}/git/refs/heads/{branch}", headers=headers, timeout=15)
    if ref_resp.status_code != 200:
        print(f"  Could not resolve branch '{branch}' for {owner_repo} (status {ref_resp.status_code}) — skipping")
        return []
    sha = ref_resp.json()['object']['sha']
    tree_resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}/git/trees/{sha}?recursive=1", headers=headers, timeout=60)
    if tree_resp.status_code != 200:
        print(f"  Could not fetch tree for {owner_repo}@{branch} (status {tree_resp.status_code}) — skipping")
        return []
    data = tree_resp.json()
    if data.get('truncated'):
        print(f"  WARNING: {owner_repo} tree was truncated by GitHub — using a partial file list")
    return data.get('tree', [])

def build_known_packages(tree):
    return {e['path'] for e in tree if e.get('type') == 'tree' and '/' not in e['path']
            and (e['path'].endswith('Pkg') or e['path'] == 'BaseTools')}

def build_known_components(tree, packages):
    known = set()
    for e in tree:
        if e.get('type') != 'tree':
            continue
        parts = e['path'].split('/')
        if not parts or parts[0] not in packages:
            continue
        if len(parts) >= 2:
            known.add('/'.join(parts[:2]))
        if len(parts) >= 3 and parts[1] == 'Library':
            known.add('/'.join(parts[:3]))
    return known

def build_known_devices(tree):
    known = set()
    for e in tree:
        if e.get('type') != 'tree':
            continue
        parts = e['path'].split('/')
        if len(parts) >= 3 and parts[0] in ('Platform', 'Silicon'):
            known.add(parts[1])
            known.add('/'.join(parts[1:3]))
    return known

KNOWN_COMPONENTS, KNOWN_DEVICES = set(), set()
if GROUND_AGAINST_REPO:
    print(f'Fetching {GITHUB_REPO} repo structure...')
    edk2_tree = fetch_repo_tree(GITHUB_REPO, GITHUB_TOKEN)
    grounded_packages = build_known_packages(edk2_tree)
    if grounded_packages:
        VALID_PACKAGES = sorted(grounded_packages) + ['Other']
        print(f'  VALID_PACKAGES grounded from real top-level dirs: {len(VALID_PACKAGES)} packages')
    else:
        print('  Could not derive packages from repo tree — keeping the config-cell fallback list')
    KNOWN_COMPONENTS = build_known_components(edk2_tree, set(VALID_PACKAGES))
    print(f'  {len(KNOWN_COMPONENTS)} known Package/Module paths indexed')

    print(f'Fetching {PLATFORMS_REPO} repo structure...')
    platforms_tree = fetch_repo_tree(PLATFORMS_REPO, GITHUB_TOKEN)
    KNOWN_DEVICES = build_known_devices(platforms_tree)
    print(f'  {len(KNOWN_DEVICES)} known vendor/board names indexed')
else:
    print('GROUND_AGAINST_REPO=False — skipping repo structure fetch; package/component/device checks unavailable')

Fetching tianocore/edk2 repo structure...
  VALID_PACKAGES grounded from real top-level dirs: 29 packages
  922 known Package/Module paths indexed
Fetching tianocore/edk2-platforms repo structure...
  142 known vendor/board names indexed


## Step 7 — Fetch per-advisory ground truth from GitHub

Two independent sources, both giving a real answer for a *specific* advisory (unlike Step 6's repo-wide tree, which only checks "is this a real path somewhere," not "is this the right one here"):

1. **The advisory's own reported affected package** — every GitHub security advisory has an "affected product" field filled in when it's created (e.g. `OvmfPkg (EDK2)`). This needs no fix commit and is available on almost every advisory — grounds `package` directly.
2. **The linked fix commit/PR**, when `references` points back into `tianocore/edk2` — its diff gives the exact files changed (→ `component`, and a more precise `package` than source #1) and how many lines (→ `patch_complexity`, threshold-classified). Only available on advisories that link a fix, but more precise when it exists — overrides source #1 for `package` when both are present.


In [20]:
# Cell 7b: Fetch per-advisory ground truth (GHSA-reported package + linked fix commit/PR diff)
COMMIT_URL_RE = re.compile(r'github\.com/([\w.-]+/[\w.-]+)/commit/([0-9a-f]{7,40})')
PULL_URL_RE   = re.compile(r'github\.com/([\w.-]+/[\w.-]+)/pull/(\d+)')

def find_fix_reference(references, repo):
    for url in references or []:
        m = COMMIT_URL_RE.search(url)
        if m and m.group(1).lower() == repo.lower():
            return 'commit', m.group(2), url
        m = PULL_URL_RE.search(url)
        if m and m.group(1).lower() == repo.lower():
            return 'pull', m.group(2), url
    return None, None, None

def fetch_commit_diff(owner_repo, sha, headers):
    resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}/commits/{sha}", headers=headers, timeout=20)
    if resp.status_code != 200:
        return None
    data = resp.json()
    files = [f.get('filename', '') for f in data.get('files', [])]
    return {'files_changed': len(files), 'lines_changed': data.get('stats', {}).get('total', 0), 'file_paths': files}

def fetch_pull_diff(owner_repo, number, headers):
    resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}/pulls/{number}", headers=headers, timeout=20)
    if resp.status_code != 200:
        return None
    data = resp.json()
    files_resp = requests.get(f"{GITHUB_API}/repos/{owner_repo}/pulls/{number}/files",
                               headers=headers, params={'per_page': 100}, timeout=20)
    file_paths = [f.get('filename', '') for f in files_resp.json()] if files_resp.status_code == 200 else []
    return {
        'files_changed': data.get('changed_files', len(file_paths)),
        'lines_changed': (data.get('additions', 0) or 0) + (data.get('deletions', 0) or 0),
        'file_paths': file_paths,
    }

def derive_component_from_paths(paths, packages):
    candidates = []
    for p in paths:
        parts = p.split('/')
        if not parts or parts[0] not in packages:
            continue
        candidates.append('/'.join(parts[:3]) if len(parts) >= 3 and parts[1] == 'Library' else '/'.join(parts[:2]))
    return Counter(candidates).most_common(1)[0][0] if candidates else ''

def classify_patch_complexity(files_changed, lines_changed, t=PATCH_COMPLEXITY_THRESHOLDS):
    if files_changed is None or lines_changed is None:
        return ''
    if files_changed <= t['simple_max_files'] and lines_changed <= t['simple_max_lines']:
        return 'simple'
    if files_changed <= t['moderate_max_files'] and lines_changed <= t['moderate_max_lines']:
        return 'moderate'
    return 'complex'

def parse_ghsa_reported_product(raw_name, valid_packages):
    """GHSA's own 'affected product' field reports names like 'OvmfPkg (EDK2)' — or, sometimes, a specific
    module too, like 'NetworkPkg/IScsiDxe (EDK2)'. Strip the '(EDK2)' suffix, then try matching the whole thing
    (no slash) or just the package segment before the slash. Returns (package_or_empty, component_or_empty) —
    component is only non-empty when GHSA's own field included a module, which is a free bonus ground-truth hit
    on advisories with no linked fix."""
    cleaned = re.sub(r'\s*\(edk2\)\s*$', '', raw_name or '', flags=re.IGNORECASE).strip()
    if not cleaned:
        return '', ''
    lookup = {v.lower(): v for v in valid_packages}
    direct = lookup.get(cleaned.lower())
    if direct:
        return direct, ''
    if '/' in cleaned:
        pkg_part = cleaned.split('/', 1)[0].strip()
        pkg_match = lookup.get(pkg_part.lower())
        if pkg_match:
            return pkg_match, cleaned   # e.g. ('NetworkPkg', 'NetworkPkg/IScsiDxe')
    return '', ''

fix_found, ghsa_pkg_found = 0, 0
if GROUND_AGAINST_REPO:
    headers = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

    for a in results:
        # Ground truth source #1: the advisory's own reported affected package (and, sometimes, module) —
        # always available whenever GitHub's advisory form was filled in with an 'affected product', no fix
        # commit required.
        parsed_products = [parse_ghsa_reported_product(p['name'], VALID_PACKAGES) for p in a.affected_packages]
        ghsa_pkgs = sorted({pkg for pkg, _ in parsed_products if pkg})
        ghsa_components = [comp for _, comp in parsed_products if comp]
        if ghsa_pkgs:
            a.ground_truth_package        = ghsa_pkgs
            a.ground_truth_package_source = 'ghsa-reported'
            ghsa_pkg_found += 1
        if ghsa_components and not a.ground_truth_component:
            a.ground_truth_component        = ghsa_components[0]   # e.g. 'NetworkPkg/IScsiDxe' — bonus, no fix commit needed
            a.ground_truth_component_source = 'ghsa-reported'

        # Ground truth source #2: the linked fix commit/PR diff — more precise (file-level), overrides source #1
        # for package, and is the ONLY source for component/patch_complexity (those need real file paths).
        ref_type, ref_id, ref_url = find_fix_reference(a.references, GITHUB_REPO)
        if not ref_type:
            continue
        diff = fetch_commit_diff(GITHUB_REPO, ref_id, headers) if ref_type == 'commit' else fetch_pull_diff(GITHUB_REPO, ref_id, headers)
        if not diff:
            continue
        a.fix_ref_url       = ref_url
        a.fix_files_changed = diff['files_changed']
        a.fix_lines_changed = diff['lines_changed']
        a.fix_file_paths    = diff['file_paths']
        diff_component = derive_component_from_paths(diff['file_paths'], set(VALID_PACKAGES))
        if diff_component:
            a.ground_truth_component = diff_component   # more precise than GHSA's reported product — overrides it
            a.ground_truth_component_source = 'fix-diff'
        diff_pkgs = sorted({p.split('/')[0] for p in diff['file_paths'] if p.split('/')[0] in VALID_PACKAGES})
        if diff_pkgs:
            a.ground_truth_package        = diff_pkgs
            a.ground_truth_package_source = 'fix-diff'
        a.ground_truth_patch_complexity = classify_patch_complexity(diff['files_changed'], diff['lines_changed'])
        fix_found += 1

    print(f'GHSA-reported affected package found for {ghsa_pkg_found}/{len(results)} advisories (ground truth source #1)')
    print(f'Linked fix commit/PR found for {fix_found}/{len(results)} advisories (ground truth source #2 — also grounds component/patch_complexity)')
else:
    print('GROUND_AGAINST_REPO=False — skipping ground-truth fetch entirely')

GHSA-reported affected package found for 12/12 advisories (ground truth source #1)
Linked fix commit/PR found for 0/12 advisories (ground truth source #2 — also grounds component/patch_complexity)


## Step 8 — LLM review: confirm or correct, then compare against ground truth

This is designed as a **triager-assist** tool, not a blind-prediction benchmark — the model sees everything a human triager would see, including GitHub's own reported severity and affected package where GitHub has one. Its job for those two fields is explicitly to confirm or correct that reported value (mirroring how a senior triager double-checks a report), not to guess blind while ignoring what's already there. For fields GitHub doesn't structure separately (`component`, `build_target`, `architecture`, `criticality_score`), the model still sees the full advisory text and extracts its best judgment from it — there's no separate reported value to confirm against for those, since GitHub doesn't capture them as distinct metadata.

The model's raw answer is stored as-is in the `llm_*` fields — it is **never overwritten**, even when ground truth from Step 6/7 disagrees with it. The comparison against ground truth happens immediately after, producing the `*_match` / `*_valid` fields, and only *then* is a resolved 'best answer' (ground truth if available, else the LLM's — now weak-signal-informed — answer) written to the plain `package`/`component`/etc. fields for convenience.


In [21]:
# Cell 8: LLM prediction
import anthropic

AI_SYSTEM = """You are an expert edk2/UEFI firmware security engineer who reviews published GitHub security advisories (CVE/GHSA).
You have deep knowledge of the tianocore/edk2 codebase, its packages, modules, and UEFI firmware.
You produce structured enhancements to help security triagers prioritize and route vulnerabilities faster.
Respond ONLY with valid JSON — no markdown, no preamble, no explanation."""

AI_PROMPT_TEMPLATE = """Review this published edk2 GitHub security advisory and produce a structured enhancement record
to assist a human triager. This report already has some structured metadata filled in by GitHub or the reporter —
your job is to confirm that metadata where it's right and correct it where it's wrong, the way a careful senior
triager double-checks a report, NOT to predict fields from scratch while ignoring what's already there. Every field
you return is a suggestion a human will review before acting on it, so state your best judgment even when unsure,
and explain your reasoning.

GHSA ID: __GHSA_ID__
CVE ID: __CVE_ID__
SUMMARY: "__SUMMARY__"
DESCRIPTION:
__DESCRIPTION__
GITHUB SEVERITY (weak signal): __GH_SEVERITY__
GITHUB REPORTED PACKAGE (weak signal): __REPORTED_PACKAGE__
CVSS VECTOR (weak signal): __CVSS_VECTOR__ (score: __CVSS_SCORE__)
CWE(S): __CWES__
AFFECTED VERSION RANGE(S): __AFFECTED_RANGES__

---

PART A — STRUCTURED FIELDS

package: Which EDK II packages are impacted?
  GitHub's advisory metadata reports __REPORTED_PACKAGE__ as the affected package. This is operator-entered when
  the advisory was created and can be incomplete, overly broad (e.g. naming a package that only contains the
  affected one), or occasionally wrong — review it against the advisory text.
  Options: __PACKAGES__
  → Return the full list you believe is actually impacted: keep the reported package if the text supports it, add
    any it's missing, drop or replace it if the text clearly points elsewhere.
  → Example: ["NetworkPkg", "MdePkg"]

package_confirmed: Does your "package" answer above match GitHub's reported package exactly (same set, nothing
  added or removed)?
  Options: Yes | No | Not reported (GitHub had no package metadata to confirm)

component: What is the specific driver, library, or module affected within the package?
  → Free text, format Package/ModuleName. Example: "CryptoPkg/TlsLib" or "NetworkPkg/Dhcp6Dxe".
  → If the specific component cannot be determined, use the package name only.

build_target: Which build targets are affected? Select all that apply.
  Options: __BUILD_TARGETS__, Not Sure
  → Return as a JSON array. If unknown, return ["Not Sure"].

architecture: Which CPU architectures are affected? Select all that apply.
  Options: __ARCHITECTURES__, Not Sure
  → Return as a JSON array. If unknown or architecture-independent, return ["Not Sure"].

affected_devices: Which specific hardware platforms, SoCs, or vendors are affected?
  → Free text. If none mentioned, write "Not specified".

impact: What is the security consequence of this vulnerability if exploited? One sentence, focused on the concrete
  consequence (arbitrary code execution, secure boot bypass, memory corruption, info disclosure), not the device.

---

PART B — MULTI-DIMENSIONAL PRIORITIZATION SCORES

severity_confirmed: GitHub's own severity rating for this advisory is __GH_SEVERITY__ (do not re-derive a rating
  from scratch — your job is only to confirm it's consistent with the advisory text and CVSS vector above).
  Options: Yes | No | Uncertain
  Criteria (UEFI firmware threat model, for your own reasoning only):
  - Critical-consistent: remotely exploitable, no privilege required, leads to code execution or secure boot bypass
  - High-consistent:     requires local/physical access or specific preconditions but leads to code execution/memory corruption
  - Moderate-consistent: information disclosure, denial of service, or unusual attacker capability required
  - Low-consistent:      minimal/theoretical impact, requires physical disassembly access, mostly defense-in-depth
  → If GitHub's rating is "(not yet rated)", set severity_confirmed to "Uncertain" and always fill severity_override.
  → "No" if the text clearly supports a different rating — fill severity_override in that case too.
  → "Uncertain" if too sparse to judge (leave severity_override empty unless GitHub was unrated).

severity_override: Only fill if severity_confirmed is "No", or GitHub's rating was "(not yet rated)". Else empty string.
  Options: __SEVERITY_OPTIONS__
  → Prefer a discrete level (Low/Moderate/High/Critical) when the text supports one.
  → Use "Assess severity using CVSS v3/v4" only when substantive but you genuinely can't support a discrete level —
    that tells a human to run a real CVSS calculation instead of trusting a guess. Default v3 unless the advisory
    already references CVSS v4-specific concepts (Safety/Automatable metrics).

patch_complexity: The complexity of a potential patch.
  Options: __PATCH_COMPLEXITY_OPTIONS__
  Criteria: simple = single-file, clear root cause; moderate = multi-file or needs investigation;
  complex = architectural/cross-package/protocol-level change or deep expertise required.

criticality_score: How critical the affected component is.
  Options: __CRITICALITY_OPTIONS__
  Criteria: high = boot-path/security-sensitive (CryptoPkg, SecurityPkg, NetworkPkg, MdePkg core, secure/measured boot);
  medium = platform-specific or optional, used by multiple vendors; low = tooling/emulator/optional/docs.

reflective_confidence (0-100): overall confidence in this enhancement, given how much the advisory text supported it.

---

PART C — PER-FIELD CONFIDENCE

In addition to the overall reflective_confidence above, rate your confidence (0-100) in EACH individual field
you provided in Part A/B, separately. A field can be confident even when the overall enhancement isn't (e.g. you
might be very sure of "package" but unsure of "criticality_score"), so judge each on its own — don't just repeat
the overall score for all of them.

---

Respond ONLY with this JSON:
{
  "package": ["<pkg1>", "<pkg2>"],
  "package_confirmed": "<Yes|No|Not reported>",
  "component": "<specific driver/library e.g. CryptoPkg/TlsLib>",
  "build_target": ["<target1>", "<target2>"],
  "architecture": ["<arch1>", "<arch2>"],
  "affected_devices": "<specific hardware platforms or Not specified>",
  "impact": "<one sentence>",
  "severity_confirmed": "<Yes|No|Uncertain>",
  "severity_override": "<one of: __SEVERITY_OPTIONS__, or empty string>",
  "severity_reason": "<one sentence>",
  "patch_complexity": "<simple|moderate|complex>",
  "patch_complexity_reason": "<one sentence>",
  "criticality_score": "<low|medium|high>",
  "criticality_reason": "<one sentence>",
  "reflective_confidence": <0-100>,
  "reflective_confidence_reason": "<one sentence>",
  "package_confidence": <0-100>,
  "component_confidence": <0-100>,
  "build_target_confidence": <0-100>,
  "architecture_confidence": <0-100>,
  "affected_devices_confidence": <0-100>,
  "impact_confidence": <0-100>,
  "severity_confidence": <0-100>,
  "patch_complexity_confidence": <0-100>,
  "criticality_confidence": <0-100>
}"""

def build_prompt(a):
    desc_text = a.description[:1500] if a.description else '(no description provided)'
    ranges = '; '.join(
        f"{p['name'] or 'edk2'} [{p['vulnerable_version_range'] or 'unspecified'}] (patched: {p['patched_versions'] or 'unspecified'})"
        for p in a.affected_packages
    ) or 'not specified'
    reported_pkg_names = [p['name'] for p in a.affected_packages if p.get('name')]
    reported_package = ', '.join(reported_pkg_names) if reported_pkg_names else 'not reported'
    return (AI_PROMPT_TEMPLATE
        .replace('__GHSA_ID__',         a.ghsa_id or 'unknown')
        .replace('__CVE_ID__',          a.cve_id or 'none assigned')
        .replace('__SUMMARY__',         a.original_summary)
        .replace('__DESCRIPTION__',     desc_text)
        .replace('__GH_SEVERITY__',     a.github_severity or '(not yet rated)')
        .replace('__REPORTED_PACKAGE__', reported_package)
        .replace('__CVSS_VECTOR__',     a.cvss_vector or 'not specified')
        .replace('__CVSS_SCORE__',      str(a.cvss_score) if a.cvss_score is not None else 'n/a')
        .replace('__CWES__',            ', '.join(a.cwe_ids) if a.cwe_ids else 'none listed')
        .replace('__AFFECTED_RANGES__', ranges)
        .replace('__PACKAGES__',        ' | '.join(VALID_PACKAGES))
        .replace('__BUILD_TARGETS__',   ' | '.join(VALID_BUILD_TARGET))
        .replace('__ARCHITECTURES__',   ' | '.join(VALID_ARCHITECTURES))
        .replace('__SEVERITY_OPTIONS__', ' | '.join(VALID_SEVERITY))
        .replace('__PATCH_COMPLEXITY_OPTIONS__', ' | '.join(VALID_PATCH_COMPLEXITY))
        .replace('__CRITICALITY_OPTIONS__', ' | '.join(VALID_CRITICALITY))
    )

def align_choices(raw_list, valid_list):
    """Case-insensitive snap-to-canonical WITHOUT dropping anything — preserves the LLM's raw prediction faithfully.
    Returns (aligned_list, unmatched_list)."""
    if not raw_list:
        return [], []
    lookup = {str(v).lower(): v for v in valid_list}
    aligned, unmatched = [], []
    for item in raw_list:
        m = lookup.get(str(item).strip().lower())
        aligned.append(m if m else str(item).strip())
        if not m:
            unmatched.append(item)
    return aligned, unmatched

def align_choice(raw, valid_list):
    if not raw:
        return '', True
    lookup = {str(v).lower(): v for v in valid_list}
    m = lookup.get(str(raw).strip().lower())
    return (m, True) if m else (str(raw).strip(), False)

def fuzzy_present(value, known_set, cutoff=0.6):
    """Is `value` a close match to something in known_set? Returns True/False/None (None = no known_set to check)."""
    if not value:
        return None
    if not known_set:
        return None
    return bool(difflib.get_close_matches(value, known_set, n=1, cutoff=cutoff))

def device_tokens_found(text, known_set):
    if not text or not known_set:
        return []
    lower_text = text.lower()
    return sorted({tok for tok in known_set if len(tok) > 3 and tok.lower() in lower_text})

def run_ai_enhancement(a, client):
    prompt = build_prompt(a)
    try:
        msg = client.messages.create(model=MODEL_NAME, max_tokens=1400, temperature=0,
                                      system=AI_SYSTEM, messages=[{'role': 'user', 'content': prompt}])
        raw = msg.content[0].text.strip()
        raw = re.sub(r'^```(?:json)?|```$', '', raw, flags=re.MULTILINE).strip()
        parsed = json.loads(raw)

        # ---- Store the LLM's raw prediction, unmodified (aligned to canonical casing only, nothing dropped) ----
        a.llm_package, pkg_bad = align_choices(parsed.get('package', []), VALID_PACKAGES)
        a.llm_package_confirmed = parsed.get('package_confirmed', '')
        a.llm_component        = parsed.get('component', '')
        a.llm_build_target, bt_bad = align_choices(parsed.get('build_target', []), VALID_BUILD_TARGET)
        a.llm_architecture, arch_bad = align_choices(parsed.get('architecture', []), VALID_ARCHITECTURES + ['Not Sure'])
        a.llm_affected_devices = parsed.get('affected_devices', '')
        a.llm_impact           = parsed.get('impact', '')
        a.llm_severity_confirmed = parsed.get('severity_confirmed', '')
        a.llm_severity_reason    = parsed.get('severity_reason', '')
        a.llm_severity_override, override_ok = align_choice(parsed.get('severity_override', ''), VALID_SEVERITY)
        a.llm_patch_complexity, patch_ok = align_choice(parsed.get('patch_complexity', ''), VALID_PATCH_COMPLEXITY)
        a.llm_patch_complexity_reason    = parsed.get('patch_complexity_reason', '')
        a.llm_criticality_score, cri_ok = align_choice(parsed.get('criticality_score', ''), VALID_CRITICALITY)
        a.llm_criticality_reason = parsed.get('criticality_reason', '')
        a.llm_reflective_confidence = int(parsed.get('reflective_confidence', 0))
        a.llm_reflective_confidence_reason = parsed.get('reflective_confidence_reason', '')

        a.llm_package_confidence          = int(parsed.get('package_confidence', 0) or 0)
        a.llm_component_confidence        = int(parsed.get('component_confidence', 0) or 0)
        a.llm_build_target_confidence     = int(parsed.get('build_target_confidence', 0) or 0)
        a.llm_architecture_confidence     = int(parsed.get('architecture_confidence', 0) or 0)
        a.llm_affected_devices_confidence = int(parsed.get('affected_devices_confidence', 0) or 0)
        a.llm_impact_confidence           = int(parsed.get('impact_confidence', 0) or 0)
        a.llm_severity_confidence         = int(parsed.get('severity_confidence', 0) or 0)
        a.llm_patch_complexity_confidence         = int(parsed.get('patch_complexity_confidence', 0) or 0)
        a.llm_criticality_confidence      = int(parsed.get('criticality_confidence', 0) or 0)

        # ---- Compare against ground truth (only where ground truth exists) ----
        if a.ground_truth_package:
            a.package_match = bool(set(p.lower() for p in a.llm_package) & set(p.lower() for p in a.ground_truth_package))
        if a.ground_truth_component:
            a.component_match = bool(a.llm_component) and bool(
                difflib.get_close_matches(a.llm_component, [a.ground_truth_component], cutoff=0.6))
        if a.ground_truth_patch_complexity:
            a.patch_complexity_match = bool(a.llm_patch_complexity) and a.llm_patch_complexity.lower() == a.ground_truth_patch_complexity.lower()
        if a.github_severity:   # GitHub had an actual rating to confirm — LLM's job was to agree or flag it
            a.severity_match = (a.llm_severity_confirmed == 'Yes')

        # ---- Weaker plausibility checks, used only where there's no independent ground truth above ----
        if not a.ground_truth_package:
            a.package_valid = None if not parsed.get('package') else len(pkg_bad) == 0
        if not a.ground_truth_component:
            a.component_valid = fuzzy_present(a.llm_component, KNOWN_COMPONENTS)
        device_hits = device_tokens_found(a.llm_affected_devices, KNOWN_DEVICES)
        a.device_valid = None if not a.llm_affected_devices or a.llm_affected_devices.strip().lower() in ('', 'not specified', 'n/a', 'none') else len(device_hits) > 0

        # ---- Resolved 'best answer' for convenience — ground truth wins when it exists ----
        a.package        = a.ground_truth_package if a.ground_truth_package else a.llm_package
        a.package_source = a.ground_truth_package_source if a.ground_truth_package else 'llm'
        a.component        = a.ground_truth_component if a.ground_truth_component else a.llm_component
        a.component_source = a.ground_truth_component_source if a.ground_truth_component else 'llm'
        a.build_target    = a.llm_build_target
        a.architecture     = a.llm_architecture
        a.affected_devices = a.llm_affected_devices
        a.impact           = a.llm_impact
        a.criticality_score  = a.llm_criticality_score
        a.criticality_reason = a.llm_criticality_reason
        if a.ground_truth_patch_complexity:
            a.patch_complexity  = a.ground_truth_patch_complexity
            a.patch_complexity_source = 'fix-diff'
            a.patch_complexity_reason = f"Actual fix changed {a.fix_files_changed} file(s), {a.fix_lines_changed} line(s) — {a.fix_ref_url}"
        else:
            a.patch_complexity  = a.llm_patch_complexity
            a.patch_complexity_source = 'llm'
            a.patch_complexity_reason = a.llm_patch_complexity_reason
        a.severity_score = a.github_severity
        if override_ok and a.llm_severity_override and (a.llm_severity_confirmed == 'No' or not a.severity_score):
            a.severity_score = a.llm_severity_override   # human-reviewable override; raw GH rating stays in github_severity

        warnings = []
        if pkg_bad:   warnings.append(f"package: not a real top-level dir: {pkg_bad}")
        if bt_bad:    warnings.append(f"build_target: unrecognized value(s): {bt_bad}")
        if arch_bad:  warnings.append(f"architecture: unrecognized value(s): {arch_bad}")
        if a.component_valid is False:
            warnings.append(f"component: no close match in edk2 repo tree — verify: {a.llm_component!r}")
        if a.device_valid is False:
            warnings.append(f"affected_devices: no known edk2-platforms match (may be legitimately out-of-tree): {a.llm_affected_devices!r}")
        if parsed.get('severity_override') and not override_ok:
            warnings.append(f"severity_override: unrecognized value kept as-is: {parsed.get('severity_override')!r}")
        if parsed.get('patch_complexity') and not patch_ok:
            warnings.append(f"patch_complexity: unrecognized value kept as-is: {parsed.get('patch_complexity')!r}")
        if parsed.get('criticality_score') and not cri_ok:
            warnings.append(f"criticality_score: unrecognized value kept as-is: {parsed.get('criticality_score')!r}")
        if a.package_match is False:
            warnings.append(f"package: disagrees with fix-diff ground truth {a.ground_truth_package}: {a.llm_package}")
        if a.component_match is False:
            warnings.append(f"component: disagrees with fix-diff ground truth {a.ground_truth_component!r}: {a.llm_component!r}")
        if a.patch_complexity_match is False:
            warnings.append(f"patch_complexity: disagrees with fix-diff ground truth {a.ground_truth_patch_complexity!r}: {a.llm_patch_complexity!r}")
        if a.severity_match is False:
            warnings.append(f"severity: LLM disagreed with GitHub's {a.github_severity!r} rating (suggested {a.llm_severity_override!r})")
        a.validation_warnings = warnings

    except Exception as exc:
        a.ai_error = str(exc)

if not RUN_AI:
    print('AI skipped (RUN_AI=False)')
elif not ANTHROPIC_KEY:
    print('No Anthropic key - skipping AI')
else:
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    print(f'Running LLM prediction on {len(results)} advisories using {MODEL_NAME}...')
    for idx, a in enumerate(results, 1):
        label = a.cve_id or a.ghsa_id
        print(f'  [{idx:>3}/{len(results)}] {label}: {a.original_summary[:60]}...')
        run_ai_enhancement(a, client)
        if idx < len(results):
            time.sleep(AI_DELAY_SEC)
    ai_err = sum(1 for r in results if r.ai_error)
    print(f'Done: {len(results) - ai_err} enhanced, {ai_err} errors')

Running LLM prediction on 12 advisories using claude-sonnet-4-6...
  [  1/12] CVE-2025-2296: Un-verified kernel bypass Secure Boot mechanism in direct bo...
  [  2/12] CVE-2024-38798: MdeModulePkg/Bus/Usb/UsbKbDxe: Uncleared password keystrokes...
  [  3/12] CVE-2024-38805: iSCSI Remote Memory Corruption and Denial of Service...
  [  4/12] CVE-2025-3770: SMM IDT privilege escalation vulnerability...
  [  5/12] CVE-2024-38797: Out of bound read in HashPeImageByType...
  [  6/12] CVE-2025-2295: Remote Memory Exposure in iSCSI DXE...
  [  7/12] CVE-2024-38796: Integer overflows in PeCoffLoaderRelocateImage...
  [  8/12] CVE-2024-1298: Temporary DoS vulnerability in FirmwarePerformancePei...
  [  9/12] CVE-2023-45229: Vulnerabilities in EDK2 NetworkPkg IP stack implementation...
  [ 10/12] CVE-2022-36763: Heap Buffer Overflow in Tcg2MeasureGptTable()...
  [ 11/12] CVE-2022-36764: Heap Buffer Overflow in Tcg2MeasurePeImage()...
  [ 12/12] CVE-2022-36765: Integer Overflow in CreateHob() coul

## Step 9 — Evaluate LLM predictions against GitHub ground truth

Since Step 8 now shows the model GitHub's reported severity and package (asking it to confirm or correct them, not guess blind), this table measures something appropriate for a **deployed assistant**: *how often does the final value line up with real GitHub data*, whether that's because the model correctly trusted a right reported value, correctly overrode a wrong one, or correctly derived it from context where nothing was reported. It is **not** a zero-shot LLM benchmark — for `severity`/`package`, some of this reflects the model faithfully carrying forward what GitHub already had, which is exactly the intended behavior for a triager-assist tool, not a flaw in the measurement. `component`/`patch_complexity` have no reported weak signal to lean on (GitHub doesn't structure those), so their `_match` numbers are closer to independent prediction accuracy, checked only where a fix commit exists (Step 7).

"Matched" means: **agreed with something real fetched from GitHub** — the advisory's own reported affected package, the actual fix commit/PR diff (`package_match`, `component_match`, `patch_complexity_match`), or GitHub's own reported severity rating (`severity_match`). Fields with no GitHub-backed ground truth at all (`build_target`, `architecture`, `criticality_score`) aren't in this table — there's nothing on GitHub to check them against.

`package_valid`/`component_valid`/`device_valid` are a separate, weaker signal — not counted here — since they only check "is this a real path/vendor somewhere in the repo," not "is this the correct one for this advisory."


In [22]:
# Cell 9: Evaluate against ground truth
def pct(n, d):
    return f'{100 * n / d:.0f}%' if d else 'n/a'

MATCH_FIELDS = [
    ('package (vs. GHSA/fix-diff)', 'package_match'),
    ('component (vs. GHSA/fix-diff)', 'component_match'),
    ('patch_complexity (vs. fix-diff)', 'patch_complexity_match'),
    ('severity (vs. GitHub rating)',  'severity_match'),
]

print(f"{'Field':<32}{'Checked':>10}{'Matched':>10}{'Match rate':>12}")
print('-' * 64)
eval_summary = {}
overall_checked, overall_matched = 0, 0
for label, attr in MATCH_FIELDS:
    values  = [getattr(r, attr) for r in results]
    checked = [v for v in values if v is not None]
    matched = [v for v in checked if v is True]
    eval_summary[label] = {'checked': len(checked), 'matched': len(matched), 'match_rate': pct(len(matched), len(checked))}
    overall_checked += len(checked); overall_matched += len(matched)
    print(f'{label:<32}{len(checked):>10}{len(matched):>10}{pct(len(matched), len(checked)):>12}')
print('-' * 64)
print(f"{'OVERALL':<32}{overall_checked:>10}{overall_matched:>10}{pct(overall_matched, overall_checked):>12}")
eval_summary['overall'] = {'checked': overall_checked, 'matched': overall_matched, 'match_rate': pct(overall_matched, overall_checked)}
print()

fix_grounded = sum(1 for r in results if r.ground_truth_patch_complexity)
ghsa_grounded = sum(1 for r in results if r.ground_truth_package_source == 'ghsa-reported')
print(f'{ghsa_grounded}/{len(results)} advisories had a GHSA-reported affected package (package ground truth, no fix commit needed).')
print(f'{fix_grounded}/{len(results)} advisories had a linked fix commit/PR (component + patch_complexity ground truth, plus a more precise package).')
print()
print('Plausibility checks (not counted above — no per-advisory ground truth, just "is this real somewhere in the repo"):')
for label, attr in [('package_valid', 'package_valid'), ('component_valid', 'component_valid'), ('device_valid', 'device_valid')]:
    values  = [getattr(r, attr) for r in results]
    checked = [v for v in values if v is not None]
    matched = [v for v in checked if v is True]
    print(f'  {label:<18}{len(checked):>4} checked, {len(matched):>4} valid ({pct(len(matched), len(checked))})')

with open('eval_summary.json', 'w', encoding='utf-8') as f:
    json.dump(eval_summary, f, indent=2)
print()
print('Saved: eval_summary.json')

Field                              Checked   Matched  Match rate
----------------------------------------------------------------
package (vs. GHSA/fix-diff)             12        12        100%
component (vs. GHSA/fix-diff)            1         1        100%
patch_complexity (vs. fix-diff)          0         0         n/a
severity (vs. GitHub rating)            12         8         67%
----------------------------------------------------------------
OVERALL                                 25        21         84%

12/12 advisories had a GHSA-reported affected package (package ground truth, no fix commit needed).
0/12 advisories had a linked fix commit/PR (component + patch_complexity ground truth, plus a more precise package).

Plausibility checks (not counted above — no per-advisory ground truth, just "is this real somewhere in the repo"):
  package_valid        0 checked,    0 valid (n/a)
  component_valid     11 checked,   10 valid (91%)
  device_valid         1 checked,    1 valid

## Step 10 — Per-advisory results

In [23]:
# Cell 10: Render per-advisory results
from IPython.display import display, HTML

def score_badge(label, value, color):
    return f"<span style='background:{color};color:#fff;padding:2px 8px;border-radius:4px;font-size:12px;font-weight:600'>{label}: {value}</span>"

def severity_color(s):
    return {'Critical': '#8e1a1a', 'High': '#c0392b', 'Moderate': '#d4860a', 'Low': '#2d9e6b',
            'Assess severity using CVSS v3': '#555', 'Assess severity using CVSS v4': '#555'}.get(s, '#888')

def patch_complexity_color(t):
    return {'complex': '#8e44ad', 'moderate': '#185fa5', 'simple': '#2d9e6b'}.get(t, '#888')

def criticality_color(c):
    return {'high': '#c0392b', 'medium': '#d4860a', 'low': '#2d9e6b'}.get(c, '#888')

def match_mark(v):
    return {True: '✓ matches GH', False: '✗ disagrees with GH', None: ''}.get(v, '')

def source_icon(source):
    # 🔗 = confirmed via a real fix commit/PR diff (strongest). 📋 = confirmed via GitHub's own reported
    # metadata, not a diff (still real ground truth, one tier softer). Nothing = LLM-only, no ground truth.
    return {'fix-diff': ' 🔗', 'ghsa-reported': ' 📋'}.get(source, '')

def conf_span(value):
    # Small additive annotation showing this specific field's own confidence (0-100), separate from the
    # overall reflective_confidence badge shown at the top of the card.
    return f"<span style='color:#bbb;font-size:10px'> (conf: {value})</span>"

def render_advisory(a, idx, total):
    if a.ai_error:
        ai_block = f"<div style='color:#c0392b;font-size:13px;margin-top:8px'>AI error: {a.ai_error}</div>"
    elif a.llm_patch_complexity or a.patch_complexity:
        ai_block = f"""
        <div style='margin-top:10px;padding:10px 14px;background:#f8f9ff;border-left:3px solid #4a6cf7;border-radius:0 6px 6px 0'>
          <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px;flex-wrap:wrap'>
            <span style='font-size:13px;font-weight:600'>Enhancement</span>
            {score_badge('Severity', a.severity_score, severity_color(a.severity_score))}
            {score_badge('Patch Complexity', a.patch_complexity + (' 🔗' if a.patch_complexity_source == 'fix-diff' else ''), patch_complexity_color(a.patch_complexity))}
            {score_badge('Criticality', a.criticality_score, criticality_color(a.criticality_score))}
            {score_badge('Confidence', f'{a.llm_reflective_confidence}/100', '#555')}
          </div>
          <div style='display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-bottom:8px'>
            <div>
              <p style='font-size:12px;font-weight:600;margin:0 0 2px'>Structured fields (LLM, 🔗 = ground truth)</p>
              <table style='font-size:12px;color:#444;border-collapse:collapse;width:100%'>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>Package</td><td>{', '.join(a.package)}{source_icon(a.package_source)} <span style='color:#999'>{match_mark(a.package_match)}</span>{f" <span style='color:#bbb;font-size:10px'>(GH reported: {a.llm_package_confirmed})</span>" if a.llm_package_confirmed else ''}{conf_span(a.llm_package_confidence)}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>Component</td><td>{a.component}{source_icon(a.component_source)} <span style='color:#999'>{match_mark(a.component_match)}</span>{conf_span(a.llm_component_confidence)}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>Build target</td><td>{', '.join(a.build_target)}{conf_span(a.llm_build_target_confidence)}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>Architecture</td><td>{', '.join(a.architecture)}{conf_span(a.llm_architecture_confidence)}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>Devices/SoCs</td><td>{a.affected_devices}{conf_span(a.llm_affected_devices_confidence)}</td></tr>
              </table>
            </div>
            <div>
              <p style='font-size:12px;font-weight:600;margin:0 0 2px'>GHSA context</p>
              <table style='font-size:12px;color:#444;border-collapse:collapse;width:100%'>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>CVE</td><td>{a.cve_id or '—'}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>GH severity</td><td>{a.github_severity or '—'}{' (via CVSS)' if a.github_severity_source == 'derived-from-cvss' else ''} <span style='color:#999'>{match_mark(a.severity_match)}</span>{conf_span(a.llm_severity_confidence)}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>CVSS</td><td>{a.cvss_score if a.cvss_score is not None else '—'}</td></tr>
                <tr><td style='padding:1px 6px 1px 0;color:#888'>CWE</td><td>{', '.join(a.cwe_ids) or '—'}</td></tr>
              </table>
            </div>
          </div>
          <p style='font-size:12px;margin:6px 0 2px'><b>Impact:</b> {a.impact}{conf_span(a.llm_impact_confidence)}</p>
          <p style='font-size:11px;color:#666;margin:2px 0'>Patch Complexity: {a.patch_complexity_reason}{conf_span(a.llm_patch_complexity_confidence)}</p>
          <p style='font-size:11px;color:#666;margin:2px 0'>Criticality: {a.criticality_reason}{conf_span(a.llm_criticality_confidence)}</p>
          {f"<p style='font-size:11px;color:#c0392b;margin:4px 0 0'>⚠ " + '; '.join(a.validation_warnings) + "</p>" if a.validation_warnings else ''}
        </div>
        """
    else:
        ai_block = "<div style='color:#888;font-size:13px;margin-top:8px'>No AI enhancement</div>"

    label = a.cve_id or a.ghsa_id
    html = f"""
    <div style='border:1px solid #e0e0e0;border-radius:8px;padding:14px 16px;margin-bottom:14px;font-family:sans-serif'>
      <div style='font-size:12px;color:#999'>Advisory {idx}/{total} · {a.published_at}</div>
      <a href='{a.url}' target='_blank' style='font-size:15px;font-weight:600;color:#222;text-decoration:none'>{label}: {a.original_summary}</a>
      {ai_block}
    </div>
    """
    display(HTML(html))

for idx, a in enumerate(results, 1):
    render_advisory(a, idx, len(results))

Package,OvmfPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 97)
Component,OvmfPkg/Library/X86QemuLoadImageLib/X86QemuLoadImageLib.c (conf: 95)
Build target,"DEBUG, RELEASE, NOOPT (conf: 55)"
Architecture,"IA32, X64 (conf: 90)"
Devices/SoCs,QEMU/KVM virtual machines using OVMF firmware with Secure Boot enabled in direct boot mode (conf: 88)
CVE,CVE-2025-2296
GH severity,High ✗ disagrees with GH (conf: 65)
CVSS,8.4
CWE,CWE-20


Package,MdeModulePkg 📋 ✓ matches GH (GH reported: Yes) (conf: 98)
Component,MdeModulePkg/Bus/Usb/UsbKbDxe (conf: 99)
Build target,"DEBUG, RELEASE, NOOPT (conf: 60)"
Architecture,Not Sure (conf: 55)
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2024-38798
GH severity,Moderate (via CVSS) ✓ matches GH (conf: 82)
CVSS,5.8
CWE,CWE-200


Package,NetworkPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 97)
Component,NetworkPkg/IScsiDxe (conf: 93)
Build target,"DEBUG, RELEASE, NOOPT (conf: 70)"
Architecture,"IA32, X64, AARCH64, ARM, RISCV64, LOONGARCH64 (conf: 75)"
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2024-38805
GH severity,Moderate (via CVSS) ✓ matches GH (conf: 85)
CVSS,6.3
CWE,—


Package,UefiCpuPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 97)
Component,UefiCpuPkg/PiSmmCpuDxeSmm (conf: 99)
Build target,"DEBUG, RELEASE, NOOPT (conf: 55)"
Architecture,X64 (conf: 95)
Devices/SoCs,Not specified (conf: 90)
CVE,CVE-2025-3770
GH severity,High ✓ matches GH (conf: 90)
CVSS,7.0
CWE,—


Package,SecurityPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 95)
Component,SecurityPkg/Library/DxeImageVerificationLib (conf: 72)
Build target,"DEBUG, RELEASE, NOOPT (conf: 60)"
Architecture,Not Sure (conf: 40)
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2024-38797
GH severity,Moderate (via CVSS) ✓ matches GH (conf: 82)
CVSS,4.6
CWE,CWE-125


...

Package,MdeModulePkg 📋 ✓ matches GH (GH reported: Yes) (conf: 98)
Component,MdeModulePkg/Universal/Acpi/FirmwarePerformanceDataTablePei/FirmwarePerformancePei.c (conf: 99)
Build target,"DEBUG, RELEASE, NOOPT (conf: 60)"
Architecture,Not Sure (conf: 50)
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2024-1298
GH severity,Moderate (via CVSS) ✓ matches GH (conf: 90)
CVSS,6.0
CWE,CWE-369


Package,NetworkPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 97)
Component,NetworkPkg/Dhcp6Dxe (conf: 55)
Build target,"DEBUG, RELEASE, NOOPT (conf: 50)"
Architecture,"IA32, X64, AARCH64, ARM, RISCV64 (conf: 60)"
Devices/SoCs,Not specified (conf: 90)
CVE,CVE-2023-45229
GH severity,High ✗ disagrees with GH (conf: 55)
CVSS,8.3
CWE,"CWE-119, CWE-125, CWE-338, CWE-835"


Package,SecurityPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 97)
Component,SecurityPkg/Library/DxeTpm2MeasureBootLib (conf: 98)
Build target,"DEBUG, RELEASE, NOOPT (conf: 60)"
Architecture,"IA32, X64, AARCH64, ARM, RISCV64 (conf: 70)"
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2022-36763
GH severity,High ✓ matches GH (conf: 90)
CVSS,7.0
CWE,CWE-680


Package,SecurityPkg 📋 ✓ matches GH (GH reported: Yes) (conf: 98)
Component,SecurityPkg/Library/DxeTpm2MeasureBootLib (conf: 99)
Build target,"DEBUG, RELEASE, NOOPT (conf: 60)"
Architecture,"IA32, X64, AARCH64, ARM, RISCV64 (conf: 65)"
Devices/SoCs,Not specified (conf: 95)
CVE,CVE-2022-36764
GH severity,High ✓ matches GH (conf: 85)
CVSS,7.0
CWE,CWE-680


Package,UefiPayloadPkg 📋 ✓ matches GH (GH reported: No) (conf: 65)
Component,MdeModulePkg/Library/PeiHobLib (conf: 60)
Build target,"DEBUG, RELEASE, NOOPT (conf: 70)"
Architecture,Not Sure (conf: 55)
Devices/SoCs,Not specified (conf: 90)
CVE,CVE-2022-36765
GH severity,High ✓ matches GH (conf: 78)
CVSS,7.0
CWE,CWE-680


## Step 11 — Export to JSON + CSV

In [24]:
# Cell 11: Export results
import csv
from dataclasses import asdict
from google.colab import files

with open('vul_enhancement_report.json', 'w', encoding='utf-8') as f:
    json.dump([asdict(a) for a in results], f, indent=2, ensure_ascii=False)
print('Saved: vul_enhancement_report.json (full detail, including every llm_*/ground_truth_*/*_match field)')

CSV_FIELDS = [
    'ghsa_id', 'cve_id', 'url', 'published_at', 'original_summary',
    'github_severity', 'github_severity_source', 'cvss_score', 'cvss_vector', 'cvss_version', 'cwe_ids',
    # LLM prediction, raw
    'llm_package', 'llm_package_confirmed', 'llm_component', 'llm_build_target', 'llm_architecture', 'llm_affected_devices', 'llm_impact',
    'llm_severity_confirmed', 'llm_severity_override', 'llm_severity_reason',
    'llm_patch_complexity', 'llm_patch_complexity_reason',
    'llm_criticality_score', 'llm_criticality_reason',
    'llm_reflective_confidence', 'llm_reflective_confidence_reason',
    'llm_package_confidence', 'llm_component_confidence', 'llm_build_target_confidence',
    'llm_architecture_confidence', 'llm_affected_devices_confidence', 'llm_impact_confidence',
    'llm_severity_confidence', 'llm_patch_complexity_confidence', 'llm_criticality_confidence',
    # Ground truth from GitHub
    'ground_truth_package', 'ground_truth_package_source', 'ground_truth_component', 'ground_truth_component_source', 'ground_truth_patch_complexity',
    'fix_ref_url', 'fix_files_changed', 'fix_lines_changed',
    # Comparison
    'package_match', 'component_match', 'patch_complexity_match', 'severity_match',
    'package_valid', 'component_valid', 'device_valid',
    # Resolved best-answer (ground truth if available, else LLM)
    'package', 'package_source', 'component', 'component_source',
    'build_target', 'architecture', 'affected_devices', 'impact',
    'severity_score', 'patch_complexity', 'patch_complexity_source', 'patch_complexity_reason',
    'criticality_score', 'criticality_reason',
    'validation_warnings', 'ai_error',
]

def j(v):
    return ' | '.join(v) if isinstance(v, list) else v

with open('vul_enhancement_report.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    for a in results:
        row = {k: j(getattr(a, k)) for k in CSV_FIELDS}
        row['cve_id'] = a.cve_id or ''
        row['cvss_score'] = a.cvss_score if a.cvss_score is not None else ''
        row['fix_files_changed'] = a.fix_files_changed if a.fix_files_changed is not None else ''
        row['fix_lines_changed'] = a.fix_lines_changed if a.fix_lines_changed is not None else ''
        writer.writerow(row)
print('Saved: vul_enhancement_report.csv (llm_* / ground_truth_* / *_match columns side by side for direct comparison)')

files.download('vul_enhancement_report.json')
files.download('vul_enhancement_report.csv')
files.download('eval_summary.json')
print('Downloads triggered')

Saved: vul_enhancement_report.json (full detail, including every llm_*/ground_truth_*/*_match field)
Saved: vul_enhancement_report.csv (llm_* / ground_truth_* / *_match columns side by side for direct comparison)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads triggered
